# Exercise 02 — Streaming, Output Control & Structured Data

Topics covered:
1. **Streaming** — display words as they arrive instead of waiting for the full response
2. **Pre-filling assistant messages** — steer Claude's response by starting it yourself
3. **Stop sequences** — force Claude to stop at a specific string
4. **Structured JSON extraction** — combine pre-fill + stop sequences to get clean JSON output

In [ ]:
import os
import json
from dotenv import load_dotenv
import anthropic

load_dotenv(dotenv_path="../.env")
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

print("Ready!")

## Part 1 — Response Streaming

Without streaming you have to wait for the **entire** response before displaying anything. With streaming, tokens arrive as soon as Claude generates them — great for user experience.

The `client.messages.stream()` context manager gives you a `text_stream` iterator and a `get_final_message()` method.

In [ ]:
# Non-streaming (waits for everything)
import time

start = time.time()
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": "Explain how rainbows form in 4 sentences."}]
)
elapsed = time.time() - start
print(f"[Non-streaming — waited {elapsed:.2f}s before seeing any output]")
print(response.content[0].text)

In [ ]:
# Streaming — tokens appear as they are generated
print("[Streaming — words appear immediately]\n")

with client.messages.stream(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": "Explain how rainbows form in 4 sentences."}]
) as stream:
    for text_chunk in stream.text_stream:
        print(text_chunk, end="", flush=True)

    # The final assembled message is available after the loop
    final_message = stream.get_final_message()

print(f"\n\n[Total input tokens: {final_message.usage.input_tokens}, "
      f"output tokens: {final_message.usage.output_tokens}]")

### Exercise 1a — Streaming with a System Prompt

Add a system prompt to the streaming request to make Claude respond as a very enthusiastic sports commentator. Ask it to describe making a cup of tea.

In [ ]:
# TODO: Add system parameter here
with client.messages.stream(
    model=MODEL,
    max_tokens=300,
    # system="...",
    messages=[{"role": "user", "content": "Describe making a cup of tea."}]
) as stream:
    for text_chunk in stream.text_stream:
        print(text_chunk, end="", flush=True)

## Part 2 — Pre-filling Assistant Messages

You can add an **incomplete assistant message** at the end of your messages list. Claude will treat it as something it already started saying and continue from exactly that point.

This is powerful for:
- Steering responses in a specific direction
- Forcing a certain opening structure
- Generating structured output (next section)

In [ ]:
# Without pre-filling — Claude decides how to frame the answer
response_no_prefill = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user", "content": "Is coffee or tea better?"}
    ]
)
print("Without pre-fill:")
print(response_no_prefill.content[0].text)
print()

In [ ]:
# With pre-filling — force Claude to argue FOR coffee
response_prefill = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user",      "content": "Is coffee or tea better?"},
        {"role": "assistant", "content": "Coffee is clearly better because"}  # pre-fill
    ]
)

# IMPORTANT: stitch the pre-fill + generated text together
prefill_text = "Coffee is clearly better because"
generated_text = response_prefill.content[0].text
full_response = prefill_text + generated_text

print("With pre-fill (stitched together):")
print(full_response)

### Exercise 2a — Steer Toward Tea

Modify the pre-fill above so Claude argues that **tea** is better instead. Remember to stitch the response.

In [ ]:
# TODO: Modify the pre-fill text to argue for tea
prefill_text = "Tea is clearly the superior choice because"  # change this

response = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user",      "content": "Is coffee or tea better?"},
        {"role": "assistant", "content": prefill_text}
    ]
)
print(prefill_text + response.content[0].text)

## Part 3 — Stop Sequences

Stop sequences tell Claude to **immediately stop generating** when it produces a specific string. The string itself is NOT included in the output.

This is useful for:
- Limiting response length to a specific point
- Extracting the first item from a list
- Working with custom delimiters

In [ ]:
# Without stop sequence — full output
response_full = client.messages.create(
    model=MODEL,
    max_tokens=100,
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas."}]
)
print("Without stop sequence:")
print(response_full.content[0].text)

In [ ]:
# With stop sequence — stop when Claude would say "five"
response_stopped = client.messages.create(
    model=MODEL,
    max_tokens=100,
    stop_sequences=["five"],
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas."}]
)
print(f"With stop_sequence=['five']:")
print(f"Output:      '{response_stopped.content[0].text}'")
print(f"Stop reason: {response_stopped.stop_reason}")

In [ ]:
# Refine the stop sequence to get cleaner output (no trailing comma/space)
response_clean = client.messages.create(
    model=MODEL,
    max_tokens=100,
    stop_sequences=[", five"],  # include the preceding comma+space
    messages=[{"role": "user", "content": "Count from 1 to 10, separated by commas."}]
)
print("Clean output (stop at ', five'):")
print(f"'{response_clean.content[0].text}'")

### Exercise 3a — Get Only the First Bullet Point

Ask Claude to list 5 reasons to exercise. Use a stop sequence so only the **first reason** is returned.

*Hint: look at what character Claude uses to start the second bullet.*

In [ ]:
# First, run without a stop sequence to see the full list and what delimiter Claude uses
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": "List 5 reasons to exercise regularly."}]
)
print(response.content[0].text)

In [ ]:
# TODO: Now add a stop sequence so only the first reason is returned
response_first_only = client.messages.create(
    model=MODEL,
    max_tokens=300,
    stop_sequences=["TODO"],  # replace with the appropriate stop sequence
    messages=[{"role": "user", "content": "List 5 reasons to exercise regularly."}]
)
print("First reason only:")
print(response_first_only.content[0].text)

## Part 4 — Structured Data Extraction

Combining **pre-filling + stop sequences** is the simplest way to extract raw JSON from Claude without getting surrounding commentary or markdown formatting.

**Pattern:**
```
User:      "Extract the data as JSON"
Assistant: "```json"          ← pre-fill (opening delimiter)
Claude:    {"name": ...}      ← only the JSON, stops at:
Stop seq:  "```"              ← closing delimiter
```

In [ ]:
# Without structured extraction — Claude adds commentary
raw_text = """John Smith, 34 years old, senior software engineer at Acme Corp.
He has been working there since 2018 and lives in San Francisco."""

response_unstructured = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": f"Extract person info as JSON:\n{raw_text}"}]
)
print("Without structured extraction:")
print(response_unstructured.content[0].text)

In [ ]:
# With pre-fill + stop sequence — clean JSON only
response_structured = client.messages.create(
    model=MODEL,
    max_tokens=300,
    stop_sequences=["```"],          # stop at closing delimiter
    messages=[
        {"role": "user",      "content": f"Extract person info as JSON.\n\nText: {raw_text}"},
        {"role": "assistant", "content": "```json\n"}  # pre-fill with opening delimiter
    ]
)
json_text = response_structured.content[0].text.strip()
print("Raw JSON output:")
print(json_text)

print("\nParsed Python dict:")
data = json.loads(json_text)
print(data)

### Exercise 4a — Extract Product Data

Use the same pre-fill + stop sequence pattern to extract product info from the text below as a JSON object with keys: `name`, `price`, `category`, `in_stock`.

In [ ]:
product_text = """The UltraBlend Pro 5000 blender is currently available at $89.99.
It belongs to the kitchen appliances category and is currently in stock with 47 units remaining."""

# TODO: Use pre-fill + stop sequence to extract the product data as JSON
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    stop_sequences=["```"],
    messages=[
        {
            "role": "user",
            "content": (
                f"Extract product info as JSON with keys: name, price, category, in_stock.\n\n"
                f"Text: {product_text}"
            )
        },
        {"role": "assistant", "content": "```json\n"}  # pre-fill
    ]
)

json_text = response.content[0].text.strip()
data = json.loads(json_text)
print("Extracted product data:")
print(json.dumps(data, indent=2))

### Exercise 4b — Extract as Python List

The same technique works for Python lists (or any other structured format). Extract the items and their quantities from the shopping list below.

In [ ]:
shopping_text = """I need to buy: 2 apples, a dozen eggs, 500g of pasta, 
one bottle of olive oil, and 3 tomatoes from the store."""

# TODO: Extract items as a JSON list of {"item": ..., "quantity": ...} objects
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    stop_sequences=["```"],
    messages=[
        {
            "role": "user",
            "content": (
                "Extract all shopping items as a JSON array. "
                "Each object should have 'item' and 'quantity' keys.\n\n"
                f"Shopping list: {shopping_text}"
            )
        },
        {"role": "assistant", "content": "```json\n"}
    ]
)

items = json.loads(response.content[0].text.strip())
print(f"Found {len(items)} items:")
for item in items:
    print(f"  - {item['quantity']} × {item['item']}")

## Summary

- ✅ **Streaming** — `client.messages.stream()` + `text_stream` for real-time output
- ✅ **Pre-filling** — add a partial assistant message to steer Claude's response
- ✅ **Stop sequences** — `stop_sequences=["..."]` to halt generation at a delimiter
- ✅ **Structured extraction** — combine pre-fill + stop to get raw JSON/code

**Next:** [03_prompt_engineering_eval.ipynb](03_prompt_engineering_eval.ipynb) — building an evaluation pipeline and applying prompt engineering techniques